# Gated fusion of the lab detector and the GenD ensemble

Two experts, each strong on one domain:

- **lab expert:** the LayerNorm-tuned CLIP detector v1 (commit b66500a), strong on DeepFakeFace photos and diffusion images;
- **video expert:** the mean of the official GenD CLIP ViT-L/14 and Perception Encoder checkpoints (MIT, trained on FaceForensics++), strong on video face swaps (0.967 Celeb-DF video AUC on the lab's crops).

A **gate**, a linear probe on GenD CLIP's normalized features, predicts whether a crop comes from the lab domain or from video, and the attack score is `gate * lab + (1 - gate) * video` (`fusion.gated_mixture`). The gate is fitted on the lab calibration side of each protocol and the FF++ validation split; Celeb-DF never enters any fit. The max of the two experts is reported alongside.

**Runtime:** Colab Pro, L4 or A100. **Needs** the Colab secret `HF_TOKEN`.

### Configuration

In [ ]:
REPO_SOURCE = "git+https://github.com/paulohenriquevn/face-decision-lab@workspace"
HF_DATASET = "paulohenriquevn/face-decision-data"
CELEBDF_DATASET = "paulohenriquevn/face-decision-celebdf-test"
FFPP_DATASET = "paulohenriquevn/face-decision-ffpp"
HF_MODEL = "paulohenriquevn/face-decision-clip-ln"  # v1 checkpoints; reports go to pretrained/<detector>/<test>.json
CLIP_MODEL = "openai/clip-vit-large-patch14"
GEND_MODELS = {"clip": "yermandy/GenD_CLIP_L_14", "pe": "yermandy/GenD_PE_L"}
GEND_FACE_SCALE = 1.3
LAB_PROTOCOLS = ("subject_holdout", "unseen_text2img", "unseen_insight", "unseen_inpainting")

### Installation

In [ ]:
%%capture
!pip install -q "face-decision-lab @ {REPO_SOURCE}" "timm>=1.0.20"

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token = userdata.get("HF_TOKEN"))

### Data

In [ ]:
from face_decision_lab.hub import load_hub_samples

samples, hub_images = load_hub_samples(HF_DATASET)
images = dict(zip([s.image_path for s in samples], hub_images, strict = True))
for prefix, repo in (("celebdf", CELEBDF_DATASET), ("faceforensics", FFPP_DATASET)):
    extra, extra_images = load_hub_samples(repo, key_prefix = prefix)
    images.update(zip([s.image_path for s in extra], extra_images, strict = True))
    samples += extra
print(len(samples), "crops")

### Experts

GenD models are built from their own `modeling_gend.py` (transformers 5 cannot `from_pretrained` them) and checked on the model card's examples. One pass returns both fake probabilities and GenD CLIP's normalized features for the gate.

In [ ]:
import importlib.util, io, json
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import requests
import torch
from huggingface_hub import HfApi, hf_hub_download
from PIL import Image
from safetensors.torch import load_file
from torch import nn
from tqdm.auto import tqdm
from transformers import CLIPImageProcessor, CLIPVisionModelWithProjection
from face_decision_lab.baselines import LinearProbe
from face_decision_lab.faces import tighten_crop
from face_decision_lab.fusion import gated_mixture, in_lab_domain
from face_decision_lab.protocol import evaluation_report, protocol_split
from face_decision_lab.rows import open_image

api = HfApi()
pool = ThreadPoolExecutor(max_workers = 8)
EXAMPLES = ["https://github.com/yermandy/deepfake-detection/blob/main/datasets/FF/DF/000_003/000.png?raw=true",
            "https://github.com/yermandy/deepfake-detection/blob/main/datasets/FF/real/000/000.png?raw=true"]
example_frames = [Image.open(io.BytesIO(requests.get(u, timeout = 30).content)).convert("RGB") for u in EXAMPLES]

def load_gend(repo):
    spec = importlib.util.spec_from_file_location(f"modeling_{repo.split('/')[1]}", hf_hub_download(repo, "modeling_gend.py"))
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    model = module.GenD(module.GenDConfig.from_pretrained(repo))
    loaded = model.load_state_dict(load_file(hf_hub_download(repo, "model.safetensors")), strict = False)
    assert not loaded.missing_keys and not loaded.unexpected_keys, loaded
    model = model.cuda().eval()
    with torch.no_grad():
        fake = model(torch.stack([model.feature_extractor.preprocess(f) for f in example_frames]).cuda()).softmax(-1)[:, 1].tolist()
    assert fake[0] > 0.5 > fake[1], (repo, fake)
    return model

gend = {name: load_gend(repo) for name, repo in GEND_MODELS.items()}

def gend_outputs(part):
    """Fake probability of each GenD model and GenD CLIP's L2-normalized features."""
    probabilities = {name: [] for name in gend}
    features = []
    with torch.no_grad(), torch.autocast("cuda", dtype = torch.bfloat16):
        for i in tqdm(range(0, len(part), 128), leave = False):
            crops = list(pool.map(lambda s: tighten_crop(open_image(images[s.image_path]), GEND_FACE_SCALE), part[i:i + 128]))
            for name, model in gend.items():
                x = torch.stack([model.feature_extractor.preprocess(c) for c in crops]).cuda()
                pooled = model.feature_extractor(x)
                probabilities[name] += model.model(pooled).float().softmax(-1)[:, 1].cpu().tolist()
                if name == "clip":
                    features.append(nn.functional.normalize(pooled.float(), dim = -1).cpu().numpy())
    return {name: np.array(p) for name, p in probabilities.items()}, np.concatenate(features)

processor = CLIPImageProcessor.from_pretrained(CLIP_MODEL)

class LayerNormTunedCLIPv1(nn.Module):  # the v1 detector, as trained at commit b66500a
    def __init__(self):
        super().__init__()
        self.encoder = CLIPVisionModelWithProjection.from_pretrained(CLIP_MODEL)
        self.head = nn.Linear(self.encoder.config.projection_dim, 1)

    def forward(self, pixels):
        return self.head(nn.functional.normalize(self.encoder(pixel_values = pixels).image_embeds, dim = -1)).squeeze(-1)

def v1_scores(checkpoint, part):
    model = LayerNormTunedCLIPv1()
    model.load_state_dict(torch.load(hf_hub_download(HF_MODEL, checkpoint)), strict = False)
    model = model.cuda().eval()
    out = []
    with torch.no_grad(), torch.autocast("cuda", dtype = torch.bfloat16):
        for i in tqdm(range(0, len(part), 128), leave = False):
            batch = list(pool.map(lambda s: open_image(images[s.image_path]), part[i:i + 128]))
            x = processor(images = batch, return_tensors = "pt")["pixel_values"].cuda()
            out += torch.sigmoid(model(x).float()).cpu().tolist()
    return np.array(out)

### Score

In [ ]:
KEYS = ("samples", "auc", "eer", "bpcer_at_max_apcer", "videos", "video_auc")

def report(detector, test, part, scores, **extra):
    result = {k: v for k, v in evaluation_report(part, list(scores)).items() if k in KEYS} | extra
    api.upload_file(path_or_fileobj = json.dumps(result, indent = 2).encode(), repo_id = HF_MODEL,
                    path_in_repo = f"pretrained/{detector}/{test}.json", commit_message = f"{detector} on {test}")
    print(detector, test, json.dumps({k: round(result[k], 4) for k in ("auc", "bpcer_at_max_apcer", "video_auc")}))

_, ffpp_val, ffpp_test = protocol_split(samples, "ffpp")
celebdf_test = protocol_split(samples, "cross_celebdf")[2]
ffpp_val_gend, ffpp_val_features = gend_outputs(ffpp_val)

for protocol in LAB_PROTOCOLS:
    _, lab_calibration, test = protocol_split(samples, protocol)
    checkpoint = f"{protocol}/trained_parameters.pt"
    _, lab_features = gend_outputs(lab_calibration)
    calibration = lab_calibration + ffpp_val
    gate = LinearProbe().fit(np.vstack([lab_features, ffpp_val_features]), in_lab_domain(calibration))
    targets = [(protocol, test)]
    if protocol == "subject_holdout":
        targets += [("cross_celebdf", celebdf_test), ("ffpp", ffpp_test)]
    for name, part in targets:
        gend_scores, features = gend_outputs(part)
        video = (gend_scores["clip"] + gend_scores["pe"]) / 2
        lab = v1_scores(checkpoint, part)
        gate_values = gate.attack_scores(features)
        context = {"v1_checkpoint": checkpoint, "mean_gate": float(gate_values.mean())}
        report("gend_ensemble_clip_pe", name, part, video)
        report("fusion_max_v1_gendens", name, part, np.maximum(lab, video), **context)
        report("fusion_gated_v1_gendens", name, part, gated_mixture(gate_values, lab, video), **context)